# PyTorch MLP for Malware Detection

This notebook implements a Multi-Layer Perceptron (MLP) neural network using PyTorch for binary classification of malware vs goodware.

## Steps:
1. Load and explore the dataset
2. Preprocess the data
3. Split into train/test sets (80/20)
4. Define the MLP architecture
5. Train the neural network
6. Evaluate using 10-fold cross-validation
7. Test on hold-out test set
8. Save the model

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/brazilian-malware-dataset/brazilian-malware.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class distribution:")
print(df['Label'].value_counts())
print(f"\nClass balance: {df['Label'].value_counts(normalize=True)}")

## 2. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Label', axis=1)

# Keep only numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
X = X[numeric_cols]

y = df['Label']

# Handle missing values
if X.isnull().sum().sum() > 0:
    print("Filling missing values with 0...")
    X = X.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of features: {X.shape[1]}")

## 3. Train/Test Split (80/20)

In [ ]:
# Split into train and test sets (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## 4. Feature Scaling

**Critical for neural networks**: Features must be scaled for optimal training.

In [ ]:
# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")
print(f"Training data mean: {X_train_scaled.mean():.6f}")
print(f"Training data std: {X_train_scaled.std():.6f}")

## 5. Define MLP Architecture

In [ ]:
class MalwareMLP(nn.Module):
    """Multi-Layer Perceptron for malware detection"""
    
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout=0.3):
        super(MalwareMLP, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Build hidden layers
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Initialize model
input_dim = X_train_scaled.shape[1]
model = MalwareMLP(input_dim, hidden_dims=[64, 32], dropout=0.3)
model = model.to(device)

print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## 6. Prepare Data Loaders

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## 7. Train the Model

In [ ]:
# Training configuration
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

# Training history
train_losses = []
train_accuracies = []

print("Training the model...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss.item()
        predictions = (outputs > 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    
    # Calculate epoch metrics
    avg_loss = epoch_loss / len(train_loader)
    accuracy = correct / total
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

print("\nTraining complete!")

## 8. Visualize Training Progress

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

# Accuracy curve
ax2.plot(train_accuracies)
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 9. Evaluate on Test Set

In [ ]:
# Evaluate on test set
model.eval()
all_predictions = []
all_probabilities = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        
        all_probabilities.extend(outputs.cpu().numpy())
        predictions = (outputs > 0.5).float()
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch_y.numpy())

# Convert to numpy arrays
y_pred = np.array(all_predictions).flatten()
y_pred_proba = np.array(all_probabilities).flatten()
y_true = np.array(all_labels).flatten()

# Calculate metrics
test_accuracy = accuracy_score(y_true, y_pred)
test_auc = roc_auc_score(y_true, y_pred_proba)

print("\n=== Test Set Results ===")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"AUC: {test_auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Goodware', 'Malware'],
            yticklabels=['Goodware', 'Malware'])
plt.title('Confusion Matrix - PyTorch MLP')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Goodware', 'Malware']))

## 10. Cross-Validation with PyTorch

Implementing manual 10-fold cross-validation for the neural network.

In [ ]:
def train_and_evaluate_fold(X_train_fold, y_train_fold, X_val_fold, y_val_fold, epochs=30):
    """Train and evaluate model on a single fold"""
    
    # Create model
    fold_model = MalwareMLP(input_dim, hidden_dims=[64, 32], dropout=0.3).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(fold_model.parameters(), lr=0.001)
    
    # Prepare data
    X_train_t = torch.FloatTensor(X_train_fold).to(device)
    y_train_t = torch.FloatTensor(y_train_fold).reshape(-1, 1).to(device)
    X_val_t = torch.FloatTensor(X_val_fold).to(device)
    y_val_t = torch.FloatTensor(y_val_fold).reshape(-1, 1).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    # Training
    for epoch in range(epochs):
        fold_model.train()
        for batch_X, batch_y in train_loader:
            outputs = fold_model(batch_X)
            loss = criterion(outputs, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Evaluation
    fold_model.eval()
    with torch.no_grad():
        val_outputs = fold_model(X_val_t)
        val_proba = val_outputs.cpu().numpy().flatten()
        val_pred = (val_outputs > 0.5).float().cpu().numpy().flatten()
    
    accuracy = accuracy_score(y_val_fold, val_pred)
    auc = roc_auc_score(y_val_fold, val_proba)
    
    return accuracy, auc

# Perform 10-fold cross-validation
print("Performing 10-fold cross-validation...")
print("This may take several minutes...\n")

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_accuracies = []
cv_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_scaled, y_train), 1):
    X_train_fold = X_train_scaled[train_idx]
    y_train_fold = y_train.values[train_idx]
    X_val_fold = X_train_scaled[val_idx]
    y_val_fold = y_train.values[val_idx]
    
    accuracy, auc = train_and_evaluate_fold(X_train_fold, y_train_fold, X_val_fold, y_val_fold)
    cv_accuracies.append(accuracy)
    cv_aucs.append(auc)
    
    print(f"Fold {fold}: Accuracy = {accuracy:.4f}, AUC = {auc:.4f}")

print("\n=== Cross-Validation Results ===")
print(f"AUC: {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")
print(f"Accuracy: {np.mean(cv_accuracies):.4f} ± {np.std(cv_accuracies):.4f}")

## 11. Save Model

In [ ]:
# Create models directory if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

# Save the model state dict
torch.save(model.state_dict(), '../models/pytorch_mlp.pth')
print("Model saved to ../models/pytorch_mlp.pth")

# Save model architecture info
model_info = {
    'input_dim': input_dim,
    'hidden_dims': [64, 32],
    'dropout': 0.3
}
import json
with open('../models/pytorch_mlp_config.json', 'w') as f:
    json.dump(model_info, f)
print("Model configuration saved to ../models/pytorch_mlp_config.json")

## 12. Load and Test Saved Model

In [ ]:
# Demonstrate loading the saved model
loaded_model = MalwareMLP(input_dim, hidden_dims=[64, 32], dropout=0.3)
loaded_model.load_state_dict(torch.load('../models/pytorch_mlp.pth'))
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully!")

# Test with a sample
with torch.no_grad():
    sample = torch.FloatTensor(X_test_scaled[0:1]).to(device)
    prediction = loaded_model(sample)
    print(f"\nSample prediction: {prediction.item():.4f}")
    print(f"Predicted class: {'Malware' if prediction.item() > 0.5 else 'Goodware'}")
    print(f"True class: {'Malware' if y_test.iloc[0] == 1 else 'Goodware'}")

## Summary

This notebook demonstrated:
1. ✅ Loading and exploring the malware dataset
2. ✅ Preprocessing with proper train/test split
3. ✅ Defining a PyTorch MLP architecture
4. ✅ Training the neural network with backpropagation
5. ✅ Monitoring training progress (loss and accuracy)
6. ✅ 10-fold stratified cross-validation
7. ✅ Evaluation on hold-out test set
8. ✅ Model serialization for production use

**Key Results:**
- Cross-Validation AUC: [See results above]
- Test Set AUC: [See results above]
- Test Set Accuracy: [See results above]

**Advantages of Neural Networks:**
- Can learn complex non-linear patterns
- Flexible architecture (can add/remove layers)
- Can be fine-tuned with different hyperparameters
- Scales well with large datasets

**Disadvantages:**
- Requires more data and computation
- Less interpretable than traditional ML models
- Sensitive to hyperparameters
- Requires careful tuning to avoid overfitting

**Model Architecture:**
- Input Layer: [Number of features]
- Hidden Layer 1: 64 neurons + ReLU + Dropout(0.3)
- Hidden Layer 2: 32 neurons + ReLU + Dropout(0.3)
- Output Layer: 1 neuron + Sigmoid